## Initializations

In [0]:
from delta import DeltaTable
from pyspark.sql.functions import (
    coalesce, col, lit, when, current_timestamp, from_utc_timestamp, datediff, sha2, concat_ws, regexp_replace,
)
from pyspark.sql import functions as F, Window

import sys
sys.path.append("../aws_utils")

import aws_utils
import transform

env = aws_utils.get_env_prefix()

## Schema, Widgets & Delete

In [0]:
partition_columns = ["BusinessDate"]
run_date = dbutils.widgets.get("run_date")
run_suffix = run_date.replace("-", "_")
unity = dbutils.widgets.get("unity")

table_name = "DSettlementExposure"
derivative_table_path = f"`{env}-cdh`.derivativeactive.dsettlementexposure"

# =============================================================================
# Schema definition & table creation
# =============================================================================

schema = (
    "SettlementExposureId string,"
    "SourceSystem string,"
    "PortfolioKey string,"
    "PortfolioName string,"
    "IngestionTimestamp timestamp,"
    "BatchId string,"
    "SettlementExposureKey string,"
    "InitialSettlementKey int,"
    "PreviouSettlementKey int,"
    "SettlementKey int,"
    "ExposureKey string,"
    "AgreementKey string,"
    "PartyKey string,"
    "CDI string,"
    "CaseCode int,"
    "DebtAmount decimal(19,4),"
    "PrincipalProposedAmount decimal(19,2),"
    "ProposedAmount decimal(19,4),"
    "RemainAmount decimal(19,4),"
    "SettlementAmount decimal(19,4),"
    "ExpectedSettlementAmount decimal(19,4),"
    "DiscountAmount decimal(19,4),"
    "DiscountPercentage decimal(19,4),"
    "BalloonAmount decimal(19,2),"
    "BucketCode int,"
    "Bucket int,"
    "SettlementTypeCode string,"
    "SettlementType string,"
    "SettlementStatusCode string,"
    "SettlementStatus string,"
    "StatusCategory string,"
    "CalculatedSettlementStatus string,"
    "CreatedDate timestamp,"
    "SubmissionDate timestamp,"
    "ApprovalDate date,"
    "ImplementationDate timestamp,"
    "ExpiredDate timestamp,"
    "CancelledDate timestamp,"
    "AutoCancelledDate timestamp,"
    "RejectedDate date,"
    "OverdueDate date,"
    "FirstInstalmentDate date,"
    "NextInstalmentDate date,"
    "PastInstalmentAmount decimal(19,4),"
    "PastInstalmentNumber int,"
    "NextInstalmentAmount decimal(19,4),"
    "NumOfaccounts int,"
    "NumOfinstalments int,"
    "TotalPaymentAmount decimal(19,4),"
    "IsActiveFlag int,"
    "Comments string,"
    "KeptPercentage decimal(19,4),"
    "IsMigratedFlag boolean,"
    "MigrationComments string,"
    "PoolName string,"
    "AddedManuallyFlag boolean,"
    "isPrimarySettlementFlag boolean,"
    "ParticipationPercentage decimal(19,4),"
    "MaturityPercentage decimal(19,4),"
    "MutiplyforLCY decimal(19,4),"
    "OtherExpense1 decimal(19,4),"
    "OtherExpense2 decimal(19,4),"
    "OtherExpense3 decimal(19,4),"
    "LastUpdatedDate timestamp,"
    "CancellationReasonCode int,"
    "CancellationReason string,"
    "DiasRFCode string,"
    "BusinessDate date,"
    "MvhoCaseFrozenInterestAmt decimal(19,4),"
    "MvhoCaseWriteOffAmt decimal(19,4),"
    "MvhoCaseProvisionAmt decimal(19,4),"
    "SettlementLastInstallmentDate date,"
    "CaseWriteOffAmtLcy decimal(19,4),"
    "StartingOutstandingBalance decimal(19,4),"
    "StartingDebtAmount decimal(19,4),"
    "StartingFrozen decimal(19,4),"
    "StartingWriteOffAmount decimal(19,4),"
    "StartingCurrentBucket int,"
    "StartingCaseDenounceStatus string,"
    "StartingHostAccountStatus string,"
    "SepsIsDownPayment boolean,"
    "SepsInstallmentAmt decimal(19,4),"
    "AgencyCode string,"
    "CtrlMovedFlag boolean,"
    "ContractNumber string,"
    "CaseFrozenInterestAmtLcy decimal(19,4),"
    "CaseProvisionAmtLcy decimal(19,4),"
    "DaysPastDue int,"
    "MltAmt decimal(19,4),"
    "PromiseCaseID string,"
    "PromiseCasePurpose string,"
    "PromiseCaseDescription string,"
    "PromiseCaseArrivalDate date,"
    "SettlementPhase string,"
    "SettlementJustification string,"
    "isPromiseActive int"
)

#================== Delete Delta Table if exists for given run_date ==================
if unity == "True":
    # Check if table exists and delete for run_date if it does
    if spark.catalog.tableExists(derivative_table_path):
        transform.delete_from_unity_table(
            spark,
            derivative_table_path,
            run_date,  # type: ignore
        )

## Run SettlementEvaluation

In [0]:
# Run SettlementEvaluation to produce calculations in:
#   - derivativeinternal.settlementevaluation_<run_date>
#   - derivativeinternal.settlementloan_<run_date>
dbutils.notebook.run("./SettlementEvaluation", timeout_seconds=3600, arguments={"run_date": run_date})

## Read Tables

In [0]:
se = spark.sql(f"""
    SELECT SettlementKey, SourceSystem, PortfolioKey, PortfolioName,
           SettlementExposureKey, ExposureKey, CDI, PartyKey,
           SettlementAmount, DiscountAmount, DiscountPercentage, ProposedAmount,
           SettlementStatusCode, SettlementStatus, SettlementTypeCode, SettlementType,
           PreviouSettlementKey, BatchId, Comments, LastUpdatedDate,
           MaturityPercentage, MutiplyforLCY,
           OtherExpense1, OtherExpense2, OtherExpense3,
           CancellationReasonCode, CancellationReason,
           AgencyCode, ContractNumber, SettlementLastInstallmentDate,
           SepsIsDownPayment, SepsInstallmentAmt,
           MvhoCaseFrozenInterestAmt, MvhoCaseWriteOffAmt, MvhoCaseProvisionAmt,
           CaseWriteOffAmtLcy, CaseFrozenInterestAmtLcy, CaseProvisionAmtLcy,
           StartingOutstandingBalance, StartingDebtAmount, StartingFrozen,
           StartingWriteOffAmount, StartingCurrentBucket,
           StartingCaseDenounceStatus, StartingHostAccountStatus,
           MigrationComments, AddedManuallyFlag, BalloonAmount, CtrlMovedFlag,
           ParticipationPercentage, PrincipalProposedAmount,
           PromiseCaseID, PromiseCasePurpose, PromiseCaseDescription,
           PromiseCaseArrivalDate, SettlementPhase, SettlementJustification,
           BusinessDate
    FROM `{env}-cdh`.primaryactive.settlementexposure
    WHERE BusinessDate = '{run_date}'
""")

ev = spark.sql(f"""
    SELECT SettlementKey, SourceSystem,
           InitialSettlementKey, PrincipalProposedAmount, RemainAmount,
           NumOfAccounts, NextInstallmentDate, NextInstallmentAmount,
           FirstInstalmentDate, ApprovalDate, RejectedDate, OverdueDate,
           DebtAmount, ExpectedSettlementAmount, SettlementBucket,
           StatusCategory, SubmissionDate, ImplementationDate,
           CancelledDate, CurrentInstallmentAmount, PastInstalmentsNum,
           TotalPaymentAmount, IsActiveFlag, KeptPercentage, DaysPastDue,
           AutoCancelledDate, EvalStatus, IsMigratedFlag, PoolName,
           LastPaymentDate, CreatedDate, ExpiredDate, NumOfinstalments,
           BalloonAmount
    FROM `{env}-cdh`.derivativeinternal.settlementevaluation_{run_suffix}
""")

sl = spark.sql(f"""
    SELECT SettlementKey, SourceSystem, ExposureKey,
           ParticipationPercentage, CaseCode, DiasNumber, AgreementKey,
           IsPrimarySettlement
    FROM `{env}-cdh`.derivativeinternal.settlementloan_{run_suffix}
""")


# Split primary by source system
se_profits = se.filter(F.col("SourceSystem") == "profits")
se_qca = se.filter(F.col("SourceSystem") == "qca")

## Transform

In [0]:
# =============================================================================
# PROFITS
# =============================================================================

se_profits_exposure = se_profits.filter(F.col("ExposureKey").isNotNull())
se_profits_agreement = se_profits.filter(F.col("ExposureKey").isNull())

# JOIN profits with eval and loan
df_profits_exposure = (
    se_profits_exposure.alias("se")
    .join(ev.alias("ev"),
          (col("se.SettlementKey") == col("ev.SettlementKey")) &
          (col("se.SourceSystem") == col("ev.SourceSystem")),
          "left")
    .join(sl.alias("sl"),
          (col("se.SettlementKey") == col("sl.SettlementKey")) &
          (col("se.SourceSystem") == col("sl.SourceSystem")) &
          (regexp_replace(col("se.ExposureKey"), "^[A-Za-z]+", "") == regexp_replace(col("sl.ExposureKey"), "^[A-Za-z]+", "")),
          "left")
)

df_profits_agreement = (
    se_profits_agreement.alias("se")
    .join(ev.alias("ev"),
          (col("se.SettlementKey") == col("ev.SettlementKey")) &
          (col("se.SourceSystem") == col("ev.SourceSystem")),
          "left")
    .join(sl.alias("sl"),
          (col("se.SettlementKey") == col("sl.SettlementKey")) &
          (col("se.SourceSystem") == col("sl.SourceSystem")),
          "left")
)

def apply_profits_select(df):
    df = (
        df
        .withColumn("SettlementExposureId",
            sha2(concat_ws(";", col("se.SettlementKey"), col("sl.ExposureKey"), col("se.SourceSystem")), 256))
    )
    return df.select(
    col("SettlementExposureId").cast("string").alias("SettlementExposureId"),
    col("se.SourceSystem").cast("string").alias("SourceSystem"),
    col("se.PortfolioKey").cast("string").alias("PortfolioKey"),
    col("se.PortfolioName").cast("string").alias("PortfolioName"),
    from_utc_timestamp(current_timestamp(), "Europe/Athens").cast("timestamp").alias("IngestionTimestamp"),
    col("se.BatchId").cast("string").alias("BatchId"),
    col("se.SettlementExposureKey").cast("string").alias("SettlementExposureKey"),
    col("se.SettlementKey").cast("int").alias("SettlementKey"),
    col("sl.ExposureKey").cast("string").alias("ExposureKey"),
    col("sl.AgreementKey").cast("string").alias("AgreementKey"),
    col("se.PartyKey").cast("string").alias("PartyKey"),
    col("se.CDI").cast("string").alias("CDI"),
    col("ev.InitialSettlementKey").cast("int").alias("InitialSettlementKey"),
    col("se.PreviouSettlementKey").cast("int").alias("PreviouSettlementKey"),
    col("se.ProposedAmount").cast("decimal(19,4)").alias("ProposedAmount"),
    col("ev.PrincipalProposedAmount").cast("decimal(19,2)").alias("PrincipalProposedAmount"),
    col("ev.RemainAmount").cast("decimal(19,4)").alias("RemainAmount"),
    col("ev.NumOfAccounts").cast("int").alias("NumOfaccounts"),
    col("ev.NextInstallmentDate").cast("date").alias("NextInstalmentDate"),
    col("ev.NextInstallmentAmount").cast("decimal(19,4)").alias("NextInstalmentAmount"),
    col("ev.FirstInstalmentDate").cast("date").alias("FirstInstalmentDate"),
    col("ev.ApprovalDate").cast("date").alias("ApprovalDate"),
    col("ev.RejectedDate").cast("date").alias("RejectedDate"),
    col("ev.OverdueDate").cast("date").alias("OverdueDate"),
    col("ev.CreatedDate").cast("timestamp").alias("CreatedDate"),
    col("ev.DebtAmount").cast("decimal(19,4)").alias("DebtAmount"),
    col("se.SettlementAmount").cast("decimal(19,4)").alias("SettlementAmount"),
    col("ev.ExpectedSettlementAmount").cast("decimal(19,4)").alias("ExpectedSettlementAmount"),
    col("se.DiscountAmount").cast("decimal(19,4)").alias("DiscountAmount"),
    col("se.DiscountPercentage").cast("decimal(19,4)").alias("DiscountPercentage"),
    lit(None).cast("int").alias("BucketCode"),
    col("ev.SettlementBucket").cast("int").alias("Bucket"),
    col("se.SettlementStatusCode").cast("string").alias("SettlementStatusCode"),
    col("se.SettlementStatus").cast("string").alias("SettlementStatus"),
    col("ev.StatusCategory").cast("string").alias("StatusCategory"),
    col("ev.EvalStatus").cast("string").alias("CalculatedSettlementStatus"),
    col("se.SettlementTypeCode").cast("string").alias("SettlementTypeCode"),
    col("se.SettlementType").cast("string").alias("SettlementType"),
    col("ev.SubmissionDate").cast("timestamp").alias("SubmissionDate"),
    col("ev.ImplementationDate").cast("timestamp").alias("ImplementationDate"),
    col("ev.ExpiredDate").cast("timestamp").alias("ExpiredDate"),
    col("ev.CancelledDate").cast("timestamp").alias("CancelledDate"),
    col("ev.AutoCancelledDate").cast("timestamp").alias("AutoCancelledDate"),
    col("ev.NumOfinstalments").cast("int").alias("NumOfinstalments"),
    col("ev.CurrentInstallmentAmount").cast("decimal(19,4)").alias("PastInstalmentAmount"),
    col("ev.PastInstalmentsNum").cast("int").alias("PastInstalmentNumber"),
    col("ev.TotalPaymentAmount").cast("decimal(19,4)").alias("TotalPaymentAmount"),
    col("ev.IsActiveFlag").cast("int").alias("IsActiveFlag"),
    col("se.Comments").cast("string").alias("Comments"),
    col("ev.KeptPercentage").cast("decimal(19,4)").alias("KeptPercentage"),
    coalesce(col("ev.DaysPastDue").cast("int"), lit(0)).alias("DaysPastDue"),
    col("sl.CaseCode").cast("int").alias("CaseCode"),
    col("sl.IsPrimarySettlement").cast("boolean").alias("isPrimarySettlementFlag"),
    col("sl.ParticipationPercentage").cast("decimal(19,4)").alias("ParticipationPercentage"),
    col("se.MaturityPercentage").cast("decimal(19,4)").alias("MaturityPercentage"),
    col("se.MutiplyforLCY").cast("decimal(19,4)").alias("MutiplyforLCY"),
    col("se.OtherExpense1").cast("decimal(19,4)").alias("OtherExpense1"),
    col("se.OtherExpense2").cast("decimal(19,4)").alias("OtherExpense2"),
    col("se.OtherExpense3").cast("decimal(19,4)").alias("OtherExpense3"),
    col("se.LastUpdatedDate").cast("timestamp").alias("LastUpdatedDate"),
    col("se.CancellationReasonCode").cast("int").alias("CancellationReasonCode"),
    col("se.CancellationReason").cast("string").alias("CancellationReason"),
    col("sl.DiasNumber").cast("string").alias("DiasRFCode"),
    col("se.AgencyCode").cast("string").alias("AgencyCode"),
    col("se.ContractNumber").cast("string").alias("ContractNumber"),
    coalesce(col("se.SettlementLastInstallmentDate").cast("date"), (col("ev.ExpiredDate").cast("date"))).alias("SettlementLastInstallmentDate"),
    col("se.SepsIsDownPayment").cast("boolean").alias("SepsIsDownPayment"),
    col("se.SepsInstallmentAmt").cast("decimal(19,4)").alias("SepsInstallmentAmt"),
    col("se.MvhoCaseFrozenInterestAmt").cast("decimal(19,4)").alias("MvhoCaseFrozenInterestAmt"),
    col("se.MvhoCaseWriteOffAmt").cast("decimal(19,4)").alias("MvhoCaseWriteOffAmt"),
    col("se.MvhoCaseProvisionAmt").cast("decimal(19,4)").alias("MvhoCaseProvisionAmt"),
    col("se.CaseWriteOffAmtLcy").cast("decimal(19,4)").alias("CaseWriteOffAmtLcy"),
    col("se.CaseFrozenInterestAmtLcy").cast("decimal(19,4)").alias("CaseFrozenInterestAmtLcy"),
    col("se.CaseProvisionAmtLcy").cast("decimal(19,4)").alias("CaseProvisionAmtLcy"),
    col("se.StartingOutstandingBalance").cast("decimal(19,4)").alias("StartingOutstandingBalance"),
    col("se.StartingDebtAmount").cast("decimal(19,4)").alias("StartingDebtAmount"),
    col("se.StartingFrozen").cast("decimal(19,4)").alias("StartingFrozen"),
    col("se.StartingWriteOffAmount").cast("decimal(19,4)").alias("StartingWriteOffAmount"),
    col("se.StartingCurrentBucket").cast("int").alias("StartingCurrentBucket"),
    col("se.StartingCaseDenounceStatus").cast("string").alias("StartingCaseDenounceStatus"),
    col("se.StartingHostAccountStatus").cast("string").alias("StartingHostAccountStatus"),
    coalesce(col("ev.IsMigratedFlag"), lit(0)).cast("boolean").alias("IsMigratedFlag"), 
    col("ev.PoolName").cast("string").alias("PoolName"),
    coalesce(col("ev.BalloonAmount"), col("se.BalloonAmount")).cast("decimal(19,2)").alias("BalloonAmount"),
    col("se.CtrlMovedFlag").cast("boolean").alias("CtrlMovedFlag"),
    when(col("se.CtrlMovedFlag") == False, coalesce(col("se.CaseProvisionAmtLcy"), lit(0))).otherwise(coalesce(col("se.MvhoCaseProvisionAmt"), lit(0))).cast("decimal(19,4)").alias("MltAmt"),
    # DNA26-1804: platforms case/phase fields (passthrough from curated settlementexposure)
    col("se.PromiseCaseID").cast("string").alias("PromiseCaseID"),
    col("se.PromiseCasePurpose").cast("string").alias("PromiseCasePurpose"),
    col("se.PromiseCaseDescription").cast("string").alias("PromiseCaseDescription"),
    col("se.PromiseCaseArrivalDate").cast("date").alias("PromiseCaseArrivalDate"),
    col("se.SettlementPhase").cast("string").alias("SettlementPhase"),
    col("se.SettlementJustification").cast("string").alias("SettlementJustification"),
    # DNA26-1804: isPromiseActive = IT TABLE_STATUS='1' perimeter. Profits rows all derive
    # from the clc_case/clc_pay_promises perimeter, so =1 for every profits settlement.
    lit(1).cast("int").alias("isPromiseActive"),
    col("se.BusinessDate").cast("date").alias("BusinessDate"),
)

# Apply function to both paths and union
result_profits = apply_profits_select(df_profits_exposure).unionByName(apply_profits_select(df_profits_agreement))

In [0]:
# =============================================================================
# QCA
# =============================================================================

df_qca = (
    se_qca.alias("se")
    .join(ev.alias("ev"),
          (col("se.SettlementKey") == col("ev.SettlementKey")) &
          (col("se.SourceSystem") == col("ev.SourceSystem")),
          "left")
    .join(sl.alias("sl"),
          (col("se.SettlementKey") == col("sl.SettlementKey")) &
          (col("se.SourceSystem") == col("sl.SourceSystem")) &
          (regexp_replace(col("se.ExposureKey"), "^[A-Za-z]+", "") == regexp_replace(col("sl.ExposureKey"), "^[A-Za-z]+", "")),
          "left")
)

df_qca = (
    df_qca
    .withColumn("SettlementExposureId",
        sha2(concat_ws(";", col("se.SettlementKey"), col("sl.ExposureKey"), col("se.SourceSystem")), 256))
)

result_qca = df_qca.select(
    col("SettlementExposureId").cast("string").alias("SettlementExposureId"),
    col("se.SourceSystem").cast("string").alias("SourceSystem"),
    col("se.PortfolioKey").cast("string").alias("PortfolioKey"),
    col("se.PortfolioName").cast("string").alias("PortfolioName"),
    from_utc_timestamp(current_timestamp(), "Europe/Athens").cast("timestamp").alias("IngestionTimestamp"),
    col("se.BatchId").cast("string").alias("BatchId"),
    col("se.SettlementExposureKey").cast("string").alias("SettlementExposureKey"),
    col("se.SettlementKey").cast("int").alias("SettlementKey"),
    col("sl.ExposureKey").cast("string").alias("ExposureKey"),
    col("sl.AgreementKey").cast("string").alias("AgreementKey"),
    col("se.PartyKey").cast("string").alias("PartyKey"),
    col("se.CDI").cast("string").alias("CDI"),
    col("ev.InitialSettlementKey").cast("int").alias("InitialSettlementKey"),
    col("se.PreviouSettlementKey").cast("int").alias("PreviouSettlementKey"),
    col("se.ProposedAmount").cast("decimal(19,4)").alias("ProposedAmount"),
    col("ev.PrincipalProposedAmount").cast("decimal(19,2)").alias("PrincipalProposedAmount"),
    col("ev.RemainAmount").cast("decimal(19,4)").alias("RemainAmount"),
    col("ev.NumOfAccounts").cast("int").alias("NumOfaccounts"),
    col("ev.NextInstallmentDate").cast("date").alias("NextInstalmentDate"),
    col("ev.NextInstallmentAmount").cast("decimal(19,4)").alias("NextInstalmentAmount"),
    col("ev.FirstInstalmentDate").cast("date").alias("FirstInstalmentDate"),
    col("ev.ApprovalDate").cast("date").alias("ApprovalDate"),
    col("ev.RejectedDate").cast("date").alias("RejectedDate"),
    col("ev.OverdueDate").cast("date").alias("OverdueDate"),
    col("ev.CreatedDate").cast("timestamp").alias("CreatedDate"),
    col("ev.DebtAmount").cast("decimal(19,4)").alias("DebtAmount"),
    col("se.SettlementAmount").cast("decimal(19,4)").alias("SettlementAmount"),
    col("ev.ExpectedSettlementAmount").cast("decimal(19,4)").alias("ExpectedSettlementAmount"),
    col("se.DiscountAmount").cast("decimal(19,4)").alias("DiscountAmount"),
    col("se.DiscountPercentage").cast("decimal(19,4)").alias("DiscountPercentage"),
    lit(None).cast("int").alias("BucketCode"),
    col("ev.SettlementBucket").cast("int").alias("Bucket"),
    col("se.SettlementStatusCode").cast("string").alias("SettlementStatusCode"),
    col("se.SettlementStatus").cast("string").alias("SettlementStatus"),
    col("ev.StatusCategory").cast("string").alias("StatusCategory"),
    col("ev.EvalStatus").cast("string").alias("CalculatedSettlementStatus"),
    col("se.SettlementTypeCode").cast("string").alias("SettlementTypeCode"),
    col("se.SettlementType").cast("string").alias("SettlementType"),
    col("ev.SubmissionDate").cast("timestamp").alias("SubmissionDate"),
    col("ev.ImplementationDate").cast("timestamp").alias("ImplementationDate"),
    col("ev.ExpiredDate").cast("timestamp").alias("ExpiredDate"),
    col("ev.CancelledDate").cast("timestamp").alias("CancelledDate"),
    col("ev.AutoCancelledDate").cast("timestamp").alias("AutoCancelledDate"),
    col("ev.NumOfinstalments").cast("int").alias("NumOfinstalments"),
    col("ev.CurrentInstallmentAmount").cast("decimal(19,4)").alias("PastInstalmentAmount"),
    col("ev.PastInstalmentsNum").cast("int").alias("PastInstalmentNumber"),
    col("ev.TotalPaymentAmount").cast("decimal(19,4)").alias("TotalPaymentAmount"),
    col("ev.IsActiveFlag").cast("int").alias("IsActiveFlag"),
    col("se.Comments").cast("string").alias("Comments"),
    col("ev.KeptPercentage").cast("decimal(19,4)").alias("KeptPercentage"),
    coalesce(col("ev.DaysPastDue").cast("int"), lit(0)).alias("DaysPastDue"),
    col("sl.CaseCode").cast("int").alias("CaseCode"),
    col("sl.IsPrimarySettlement").cast("boolean").alias("isPrimarySettlementFlag"),
    coalesce(col("se.ParticipationPercentage"), col("sl.ParticipationPercentage")).cast("decimal(19,4)").alias("ParticipationPercentage"),
    col("se.MaturityPercentage").cast("decimal(19,4)").alias("MaturityPercentage"),
    col("se.MutiplyforLCY").cast("decimal(19,4)").alias("MutiplyforLCY"),
    col("se.OtherExpense1").cast("decimal(19,4)").alias("OtherExpense1"),
    col("se.OtherExpense2").cast("decimal(19,4)").alias("OtherExpense2"),
    col("se.OtherExpense3").cast("decimal(19,4)").alias("OtherExpense3"),
    col("se.LastUpdatedDate").cast("timestamp").alias("LastUpdatedDate"),
    col("se.CancellationReasonCode").cast("int").alias("CancellationReasonCode"),
    col("se.CancellationReason").cast("string").alias("CancellationReason"),
    col("sl.DiasNumber").cast("string").alias("DiasRFCode"),
    col("se.AgencyCode").cast("string").alias("AgencyCode"),
    col("se.ContractNumber").cast("string").alias("ContractNumber"),
    coalesce(col("se.SettlementLastInstallmentDate").cast("date"), (col("ev.ExpiredDate").cast("date"))).alias("SettlementLastInstallmentDate"),
    col("se.SepsIsDownPayment").cast("boolean").alias("SepsIsDownPayment"),
    col("se.SepsInstallmentAmt").cast("decimal(19,4)").alias("SepsInstallmentAmt"),
    col("se.MvhoCaseFrozenInterestAmt").cast("decimal(19,4)").alias("MvhoCaseFrozenInterestAmt"),
    col("se.MvhoCaseWriteOffAmt").cast("decimal(19,4)").alias("MvhoCaseWriteOffAmt"),
    col("se.MvhoCaseProvisionAmt").cast("decimal(19,4)").alias("MvhoCaseProvisionAmt"),
    col("se.CaseWriteOffAmtLcy").cast("decimal(19,4)").alias("CaseWriteOffAmtLcy"),
    col("se.CaseFrozenInterestAmtLcy").cast("decimal(19,4)").alias("CaseFrozenInterestAmtLcy"),
    col("se.CaseProvisionAmtLcy").cast("decimal(19,4)").alias("CaseProvisionAmtLcy"),
    col("se.StartingOutstandingBalance").cast("decimal(19,4)").alias("StartingOutstandingBalance"),
    col("se.StartingDebtAmount").cast("decimal(19,4)").alias("StartingDebtAmount"),
    col("se.StartingFrozen").cast("decimal(19,4)").alias("StartingFrozen"),
    col("se.StartingWriteOffAmount").cast("decimal(19,4)").alias("StartingWriteOffAmount"),
    col("se.StartingCurrentBucket").cast("int").alias("StartingCurrentBucket"),
    col("se.StartingCaseDenounceStatus").cast("string").alias("StartingCaseDenounceStatus"),
    col("se.StartingHostAccountStatus").cast("string").alias("StartingHostAccountStatus"),
    coalesce(col("ev.IsMigratedFlag"), lit(0)).cast("boolean").alias("IsMigratedFlag"),
    col("ev.PoolName").cast("string").alias("PoolName"),
    coalesce(col("ev.BalloonAmount"), col("se.BalloonAmount")).cast("decimal(19,2)").alias("BalloonAmount"),
    col("se.CtrlMovedFlag").cast("boolean").alias("CtrlMovedFlag"),
    when(col("se.CtrlMovedFlag") == False, coalesce(col("se.CaseProvisionAmtLcy"), lit(0))).otherwise(coalesce(col("se.MvhoCaseProvisionAmt"), lit(0))).cast("decimal(19,4)").alias("MltAmt"),
    # DNA26-1804: platforms case/phase fields — profits-only; NULL for qca (no profits clc source)
    col("se.PromiseCaseID").cast("string").alias("PromiseCaseID"),
    col("se.PromiseCasePurpose").cast("string").alias("PromiseCasePurpose"),
    col("se.PromiseCaseDescription").cast("string").alias("PromiseCaseDescription"),
    col("se.PromiseCaseArrivalDate").cast("date").alias("PromiseCaseArrivalDate"),
    col("se.SettlementPhase").cast("string").alias("SettlementPhase"),
    col("se.SettlementJustification").cast("string").alias("SettlementJustification"),
    # isPromiseActive is a profits/IT-perimeter concept — NULL for qca
    lit(None).cast("int").alias("isPromiseActive"),
    col("se.BusinessDate").cast("date").alias("BusinessDate"),
)

In [0]:
# Union profits + QCA 
result = result_profits.unionByName(result_qca)

# Write to derivative table
result.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(derivative_table_path)

In [0]:
# Cleanup of old intermediary settlement tables.
# Drop any tables whose latest ingestion happened on a day before today.
schema = f"`{env}-cdh`.derivativeinternal"
prefixes = ("settlementevaluation_", "settlementloan_")

today = (
    spark.range(1)
    .select(from_utc_timestamp(current_timestamp(), "Europe/Athens").cast("date").alias("d"))
    .collect()[0]["d"]
)

tables = spark.sql(f"SHOW TABLES IN {schema}").collect()
intermediary = [
    r["tableName"] for r in tables
    if r["tableName"].startswith(prefixes)
]

for tablename in intermediary:
    fullname = f"{schema}.{tablename}"
    try:
        last_ingested = (
            spark.table(fullname)
            .agg(F.max("IngestionTimestamp").alias("_last"))
            .collect()[0]["_last"]
        )
    except Exception as e:
        print(f"Skipping {fullname}: could not read IngestionTimestamp ({e})")
        continue
    if last_ingested is None or last_ingested.date() < today:
        spark.sql(f"DROP TABLE IF EXISTS {fullname}")
        print(f"Dropped old intermediary table {fullname} (last ingested: {last_ingested})")
    else:
        print(f"Kept {fullname} (ingested today: {last_ingested})")